In [2]:
import gc
import math
import os
import random
import re
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
SEEDS = [11, 22, 33, 44, 55]
PARTITION_SEED = 2026
SAVE_TEST_SETS = True
LABEL_COLUMN = 'label'
BINARY_COLUMN = 'binary_label'
SOURCE_COLUMN = 'source'
ORIGINAL_LABEL_COLUMN = 'original_label'
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None
EPOCHS = {'alzughabi_pso_dnn': 100, 'wang_cnn': 50, 'wang_rnn': 50, 'wang_lstm': 50, 'anwar_lightweight_1dcnn': 20}
BATCH_SIZES = {'alzughabi_pso_dnn': 2048, 'wang_cnn': 2048, 'wang_rnn': 2048, 'wang_lstm': 2048, 'anwar_lightweight_1dcnn': 2048}
MODEL_KEYS = ['alzughabi_pso_dnn', 'wang_cnn', 'wang_rnn', 'wang_lstm', 'anwar_lightweight_1dcnn', 'ensemble_feature_rf']
RESULTS_DIR = Path('/content/drive/MyDrive/Array_submission/Results/CSE_CIC_IDS2018/Experiment4_model_specific_test_ratios')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Results directory:', RESULTS_DIR)


TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Results directory: /content/drive/MyDrive/Array_submission/Results/CSE_CIC_IDS2018/Experiment4_model_specific_test_ratios


In [3]:
def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def binary_y(frame):
    return frame[BINARY_COLUMN].to_numpy(dtype=np.int8)

def optional_sample(frame, maximum_rows, seed):
    if maximum_rows is None or len(frame) <= maximum_rows:
        return frame.reset_index(drop=True).copy()
    sampled, _ = train_test_split(frame, train_size=maximum_rows, random_state=seed, shuffle=True, stratify=frame[BINARY_COLUMN])
    return sampled.reset_index(drop=True)

def prepare_test(frame, seed):
    return optional_sample(frame, MAX_TEST_ROWS, seed)

def standard_callbacks(monitor='val_loss', mode='min', patience=5):
    return [tf.keras.callbacks.EarlyStopping(monitor=monitor, mode=mode, patience=patience, restore_best_weights=True, min_delta=0.0001, verbose=1), tf.keras.callbacks.ReduceLROnPlateau(monitor=monitor, mode=mode, factor=0.5, patience=max(2, patience // 2), min_lr=1e-07, verbose=1)]


In [4]:
CSE2018_FEATURE_COLUMNS = ['Dst Port', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWR Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
CIC_TO_CSE2018_ALIASES = {'Destination Port': 'Dst Port', 'dst_port': 'Dst Port', 'protocol': 'Protocol', 'flow_duration': 'Flow Duration', 'Total Fwd Packets': 'Tot Fwd Pkts', 'Total Fwd Packet': 'Tot Fwd Pkts', 'tot_fwd_pkts': 'Tot Fwd Pkts', 'Total Backward Packets': 'Tot Bwd Pkts', 'Total Bwd packets': 'Tot Bwd Pkts', 'tot_bwd_pkts': 'Tot Bwd Pkts', 'Total Length of Fwd Packets': 'TotLen Fwd Pkts', 'Total Length of Fwd Packet': 'TotLen Fwd Pkts', 'totlen_fwd_pkts': 'TotLen Fwd Pkts', 'Total Length of Bwd Packets': 'TotLen Bwd Pkts', 'Total Length of Bwd Packet': 'TotLen Bwd Pkts', 'totlen_bwd_pkts': 'TotLen Bwd Pkts', 'Fwd Packet Length Max': 'Fwd Pkt Len Max', 'fwd_pkt_len_max': 'Fwd Pkt Len Max', 'Fwd Packet Length Min': 'Fwd Pkt Len Min', 'fwd_pkt_len_min': 'Fwd Pkt Len Min', 'Fwd Packet Length Mean': 'Fwd Pkt Len Mean', 'fwd_pkt_len_mean': 'Fwd Pkt Len Mean', 'Fwd Packet Length Std': 'Fwd Pkt Len Std', 'fwd_pkt_len_std': 'Fwd Pkt Len Std', 'Bwd Packet Length Max': 'Bwd Pkt Len Max', 'bwd_pkt_len_max': 'Bwd Pkt Len Max', 'Bwd Packet Length Min': 'Bwd Pkt Len Min', 'bwd_pkt_len_min': 'Bwd Pkt Len Min', 'Bwd Packet Length Mean': 'Bwd Pkt Len Mean', 'bwd_pkt_len_mean': 'Bwd Pkt Len Mean', 'Bwd Packet Length Std': 'Bwd Pkt Len Std', 'bwd_pkt_len_std': 'Bwd Pkt Len Std', 'Flow Bytes/s': 'Flow Byts/s', 'flow_byts_s': 'Flow Byts/s', 'Flow Packets/s': 'Flow Pkts/s', 'flow_pkts_s': 'Flow Pkts/s', 'flow_iat_mean': 'Flow IAT Mean', 'flow_iat_std': 'Flow IAT Std', 'flow_iat_max': 'Flow IAT Max', 'flow_iat_min': 'Flow IAT Min', 'Fwd IAT Total': 'Fwd IAT Tot', 'fwd_iat_tot': 'Fwd IAT Tot', 'fwd_iat_mean': 'Fwd IAT Mean', 'fwd_iat_std': 'Fwd IAT Std', 'fwd_iat_max': 'Fwd IAT Max', 'fwd_iat_min': 'Fwd IAT Min', 'Bwd IAT Total': 'Bwd IAT Tot', 'bwd_iat_tot': 'Bwd IAT Tot', 'bwd_iat_mean': 'Bwd IAT Mean', 'bwd_iat_std': 'Bwd IAT Std', 'bwd_iat_max': 'Bwd IAT Max', 'bwd_iat_min': 'Bwd IAT Min', 'fwd_psh_flags': 'Fwd PSH Flags', 'bwd_psh_flags': 'Bwd PSH Flags', 'fwd_urg_flags': 'Fwd URG Flags', 'bwd_urg_flags': 'Bwd URG Flags', 'Fwd Header Length': 'Fwd Header Len', 'fwd_header_len': 'Fwd Header Len', 'Bwd Header Length': 'Bwd Header Len', 'bwd_header_len': 'Bwd Header Len', 'Fwd Packets/s': 'Fwd Pkts/s', 'fwd_pkts_s': 'Fwd Pkts/s', 'Bwd Packets/s': 'Bwd Pkts/s', 'bwd_pkts_s': 'Bwd Pkts/s', 'Min Packet Length': 'Pkt Len Min', 'Packet Length Min': 'Pkt Len Min', 'pkt_len_min': 'Pkt Len Min', 'Max Packet Length': 'Pkt Len Max', 'Packet Length Max': 'Pkt Len Max', 'pkt_len_max': 'Pkt Len Max', 'Packet Length Mean': 'Pkt Len Mean', 'pkt_len_mean': 'Pkt Len Mean', 'Packet Length Std': 'Pkt Len Std', 'pkt_len_std': 'Pkt Len Std', 'Packet Length Variance': 'Pkt Len Var', 'pkt_len_var': 'Pkt Len Var', 'FIN Flag Count': 'FIN Flag Cnt', 'fin_flag_cnt': 'FIN Flag Cnt', 'SYN Flag Count': 'SYN Flag Cnt', 'syn_flag_cnt': 'SYN Flag Cnt', 'RST Flag Count': 'RST Flag Cnt', 'rst_flag_cnt': 'RST Flag Cnt', 'PSH Flag Count': 'PSH Flag Cnt', 'psh_flag_cnt': 'PSH Flag Cnt', 'ACK Flag Count': 'ACK Flag Cnt', 'ack_flag_cnt': 'ACK Flag Cnt', 'URG Flag Count': 'URG Flag Cnt', 'urg_flag_cnt': 'URG Flag Cnt', 'CWE Flag Count': 'CWR Flag Count', 'cwr_flag_count': 'CWR Flag Count', 'ECE Flag Count': 'ECE Flag Cnt', 'ece_flag_cnt': 'ECE Flag Cnt', 'down_up_ratio': 'Down/Up Ratio', 'Average Packet Size': 'Pkt Size Avg', 'pkt_size_avg': 'Pkt Size Avg', 'Avg Fwd Segment Size': 'Fwd Seg Size Avg', 'fwd_seg_size_avg': 'Fwd Seg Size Avg', 'Avg Bwd Segment Size': 'Bwd Seg Size Avg', 'bwd_seg_size_avg': 'Bwd Seg Size Avg', 'Fwd Avg Bytes/Bulk': 'Fwd Byts/b Avg', 'Fwd Bytes/Bulk Avg': 'Fwd Byts/b Avg', 'fwd_byts_b_avg': 'Fwd Byts/b Avg', 'Fwd Avg Packets/Bulk': 'Fwd Pkts/b Avg', 'Fwd Packet/Bulk Avg': 'Fwd Pkts/b Avg', 'fwd_pkts_b_avg': 'Fwd Pkts/b Avg', 'Fwd Avg Bulk Rate': 'Fwd Blk Rate Avg', 'Fwd Bulk Rate Avg': 'Fwd Blk Rate Avg', 'fwd_blk_rate_avg': 'Fwd Blk Rate Avg', 'Bwd Avg Bytes/Bulk': 'Bwd Byts/b Avg', 'Bwd Bytes/Bulk Avg': 'Bwd Byts/b Avg', 'bwd_byts_b_avg': 'Bwd Byts/b Avg', 'Bwd Avg Packets/Bulk': 'Bwd Pkts/b Avg', 'Bwd Packet/Bulk Avg': 'Bwd Pkts/b Avg', 'bwd_pkts_b_avg': 'Bwd Pkts/b Avg', 'Bwd Avg Bulk Rate': 'Bwd Blk Rate Avg', 'Bwd Bulk Rate Avg': 'Bwd Blk Rate Avg', 'bwd_blk_rate_avg': 'Bwd Blk Rate Avg', 'Subflow Fwd Packets': 'Subflow Fwd Pkts', 'subflow_fwd_pkts': 'Subflow Fwd Pkts', 'Subflow Fwd Bytes': 'Subflow Fwd Byts', 'subflow_fwd_byts': 'Subflow Fwd Byts', 'Subflow Bwd Packets': 'Subflow Bwd Pkts', 'subflow_bwd_pkts': 'Subflow Bwd Pkts', 'Subflow Bwd Bytes': 'Subflow Bwd Byts', 'subflow_bwd_byts': 'Subflow Bwd Byts', 'Init_Win_bytes_forward': 'Init Fwd Win Byts', 'FWD Init Win Bytes': 'Init Fwd Win Byts', 'init_fwd_win_byts': 'Init Fwd Win Byts', 'Init_Win_bytes_backward': 'Init Bwd Win Byts', 'Bwd Init Win Bytes': 'Init Bwd Win Byts', 'init_bwd_win_byts': 'Init Bwd Win Byts', 'act_data_pkt_fwd': 'Fwd Act Data Pkts', 'Fwd Act Data Pkts': 'Fwd Act Data Pkts', 'fwd_act_data_pkts': 'Fwd Act Data Pkts', 'min_seg_size_forward': 'Fwd Seg Size Min', 'fwd_seg_size_min': 'Fwd Seg Size Min', 'active_mean': 'Active Mean', 'active_std': 'Active Std', 'active_max': 'Active Max', 'active_min': 'Active Min', 'idle_mean': 'Idle Mean', 'idle_std': 'Idle Std', 'idle_max': 'Idle Max', 'idle_min': 'Idle Min', 'Fwd Segment Size Avg': 'Fwd Seg Size Avg', 'Bwd Segment Size Avg': 'Bwd Seg Size Avg'}


In [5]:
def numeric_matrix(frame, feature_columns):
    matrix = frame.reindex(columns=feature_columns).copy()
    for column in feature_columns:
        matrix[column] = pd.to_numeric(matrix[column], errors='coerce')
    return matrix.replace([np.inf, -np.inf], np.nan)

def fit_numeric_preprocessor(train_frame, feature_columns, scaler_kind, drop_constant=True):
    matrix = numeric_matrix(train_frame, feature_columns)
    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)
    active_features = list(feature_columns)
    if drop_constant:
        active_features = [column for column in feature_columns if matrix[column].nunique(dropna=False) > 1]
    if not active_features:
        raise ValueError('No non-constant training features remain.')
    matrix = matrix[active_features]
    if scaler_kind == 'standard':
        scaler = StandardScaler()
    elif scaler_kind == 'minmax':
        scaler = MinMaxScaler(feature_range=(0, 1))
    elif scaler_kind == 'none':
        scaler = None
    else:
        raise ValueError(f'Unknown scaler kind: {scaler_kind}')
    transformed = scaler.fit_transform(matrix) if scaler is not None else matrix.to_numpy()
    return ({'feature_columns': active_features, 'medians': medians, 'scaler': scaler, 'scaler_kind': scaler_kind}, np.asarray(transformed, dtype=np.float32))

def transform_numeric_preprocessor(fitted, frame):
    matrix = numeric_matrix(frame, fitted['feature_columns'])
    medians = fitted['medians'].reindex(fitted['feature_columns']).fillna(0.0)
    matrix = matrix.fillna(medians)
    transformed = fitted['scaler'].transform(matrix) if fitted['scaler'] is not None else matrix.to_numpy()
    return np.asarray(transformed, dtype=np.float32)


In [6]:
def calculate_binary_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if tn + fp > 0 else np.nan
    metrics = {'n_test': int(len(y_true)), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'accuracy': accuracy_score(y_true, y_pred), 'balanced_accuracy': balanced_accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0), 'recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0), 'specificity': specificity, 'f1': f1_score(y_true, y_pred, pos_label=1, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0), 'mcc': matthews_corrcoef(y_true, y_pred), 'false_positive_rate': fp / (fp + tn) if fp + tn > 0 else np.nan, 'roc_auc': np.nan}
    if y_score is not None and len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = roc_auc_score(y_true, np.asarray(y_score))
    return metrics

def category_recall_table(frame, y_true, y_pred):
    categories = frame[LABEL_COLUMN].astype(str).to_numpy()
    rows = []
    for category in sorted(pd.unique(categories)):
        mask = categories == category
        category_true = y_true[mask]
        category_prediction = y_pred[mask]
        rows.append({'test_category': category, 'support': int(mask.sum()), 'attack_recall': recall_score(category_true, category_prediction, pos_label=1, zero_division=0) if np.any(category_true == 1) else np.nan, 'benign_specificity': recall_score(category_true, category_prediction, pos_label=0, zero_division=0) if np.any(category_true == 0) else np.nan})
    return pd.DataFrame(rows)

def subtype_recall_table(frame, y_true, y_pred):
    subtypes = frame[ORIGINAL_LABEL_COLUMN].astype(str).str.strip().str.rstrip('.').to_numpy()
    rows = []
    for subtype in sorted(pd.unique(subtypes)):
        mask = subtypes == subtype
        subtype_true = y_true[mask]
        subtype_prediction = y_pred[mask]
        rows.append({'test_subtype': subtype, 'support': int(mask.sum()), 'attack_recall': recall_score(subtype_true, subtype_prediction, pos_label=1, zero_division=0) if np.any(subtype_true == 1) else np.nan, 'benign_specificity': recall_score(subtype_true, subtype_prediction, pos_label=0, zero_division=0) if np.any(subtype_true == 0) else np.nan})
    return pd.DataFrame(rows)


In [7]:
ALZUGHABI_SELECTED_FEATURES = ['Dst Port', 'Fwd Seg Size Min', 'Init Fwd Win Byts', 'Fwd Pkt Len Max', 'TotLen Fwd Pkts', 'Subflow Fwd Byts', 'Fwd Pkt Len Mean', 'Fwd IAT Tot', 'Flow Duration', 'Fwd IAT Mean', 'Fwd IAT Max', 'Flow IAT Max', 'Fwd Seg Size Avg', 'Flow IAT Min', 'Fwd Pkts/s', 'Flow IAT Mean', 'Fwd IAT Min', 'Flow Pkts/s', 'Fwd Header Len', 'Init Bwd Win Byts', 'Bwd Pkts/s', 'Pkt Len Max', 'Tot Fwd Pkts', 'Subflow Fwd Pkts']

def build_alzughabi_pso_dnn():
    learning_rate_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=0.001, decay_steps=10000, decay_rate=0.96, staircase=True)
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(24,)), tf.keras.layers.Dense(15, activation='softsign'), tf.keras.layers.Dense(11, activation='relu'), tf.keras.layers.Dense(2, activation='softmax')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate_schedule), loss=tf.keras.losses.BinaryCrossentropy(), metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')])
    return model

def fit_alzughabi_pso_dnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_numeric_preprocessor(train_frame, ALZUGHABI_SELECTED_FEATURES, scaler_kind='standard', drop_constant=False)
    y_train = tf.keras.utils.to_categorical(binary_y(train_frame), num_classes=2)
    model = build_alzughabi_pso_dnn()
    model.fit(X_train, y_train, validation_split=0.1, batch_size=BATCH_SIZES['alzughabi_pso_dnn'], epochs=EPOCHS['alzughabi_pso_dnn'], shuffle=True, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=8, restore_best_weights=True, min_delta=0.0001, verbose=1)], verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_alzughabi_pso_dnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_numeric_preprocessor(fitted['preprocessor'], evaluated)
    probabilities = fitted['model'].predict(X_test, batch_size=BATCH_SIZES['alzughabi_pso_dnn'], verbose=0)
    prediction = np.argmax(probabilities, axis=1).astype(np.int8)
    return (binary_y(evaluated), prediction, probabilities[:, 1], evaluated)


In [8]:
WANG_ALL_ZERO_FEATURES = {'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg'}
WANG_CANDIDATE_FEATURES = [column for column in CSE2018_FEATURE_COLUMNS if column != 'Dst Port' and column not in WANG_ALL_ZERO_FEATURES]

def fit_wang_preprocessor(train_frame):
    matrix = numeric_matrix(train_frame, WANG_CANDIDATE_FEATURES)
    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)
    active_features = [column for column in WANG_CANDIDATE_FEATURES if matrix[column].nunique(dropna=False) > 1]
    if len(active_features) > 69:
        active_features = active_features[:69]
    if not active_features:
        raise ValueError('No Wang features remain after training-only filtering.')
    scaler = StandardScaler()
    standardized = scaler.fit_transform(matrix[active_features]).astype(np.float32)
    training_minimums = standardized.min(axis=0, keepdims=True)
    nonnegative_shifts = np.where(training_minimums < 0, -training_minimums, 0.0).astype(np.float32)
    transformed = np.log1p(np.maximum(standardized + nonnegative_shifts, 0.0)).astype(np.float32)
    return ({'feature_columns': active_features, 'medians': medians, 'scaler': scaler, 'nonnegative_shifts': nonnegative_shifts}, transformed)

def transform_wang_preprocessor(fitted, frame):
    matrix = numeric_matrix(frame, fitted['feature_columns'])
    matrix = matrix.fillna(fitted['medians'].reindex(fitted['feature_columns']).fillna(0.0))
    standardized = fitted['scaler'].transform(matrix).astype(np.float32)
    return np.log1p(np.maximum(standardized + fitted['nonnegative_shifts'], 0.0)).astype(np.float32)


In [9]:
def build_wang_cnn(n_features):
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(n_features, 1)), tf.keras.layers.Conv1D(128, kernel_size=2, padding='same', activation='relu'), tf.keras.layers.MaxPooling1D(pool_size=2), tf.keras.layers.Conv1D(128, kernel_size=2, padding='same', activation='relu'), tf.keras.layers.MaxPooling1D(pool_size=2), tf.keras.layers.Conv1D(256, kernel_size=2, padding='same', activation='relu'), tf.keras.layers.MaxPooling1D(pool_size=2), tf.keras.layers.Conv1D(256, kernel_size=2, padding='same', activation='relu'), tf.keras.layers.MaxPooling1D(pool_size=2), tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.2), tf.keras.layers.Flatten(), tf.keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    return model

def fit_wang_cnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_wang_preprocessor(train_frame)
    model = build_wang_cnn(X_train.shape[1])
    model.fit(X_train[..., np.newaxis], binary_y(train_frame), validation_split=0.1, batch_size=BATCH_SIZES['wang_cnn'], epochs=EPOCHS['wang_cnn'], shuffle=True, callbacks=standard_callbacks(patience=5), verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_wang_cnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_wang_preprocessor(fitted['preprocessor'], evaluated)
    probability = fitted['model'].predict(X_test[..., np.newaxis], batch_size=BATCH_SIZES['wang_cnn'], verbose=0).reshape(-1)
    prediction = (probability >= 0.5).astype(np.int8)
    return (binary_y(evaluated), prediction, probability, evaluated)


In [10]:
def build_wang_rnn(n_features):
    model_input = tf.keras.layers.Input(shape=(1, n_features))
    x = model_input
    for _ in range(4):
        x = tf.keras.layers.SimpleRNN(128, return_sequences=True)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.SimpleRNN(256, return_sequences=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(model_input, output)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    return model

def fit_wang_rnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_wang_preprocessor(train_frame)
    model = build_wang_rnn(X_train.shape[1])
    model.fit(X_train[:, np.newaxis, :], binary_y(train_frame), validation_split=0.1, batch_size=BATCH_SIZES['wang_rnn'], epochs=EPOCHS['wang_rnn'], shuffle=True, callbacks=standard_callbacks(patience=5), verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_wang_rnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_wang_preprocessor(fitted['preprocessor'], evaluated)
    probability = fitted['model'].predict(X_test[:, np.newaxis, :], batch_size=BATCH_SIZES['wang_rnn'], verbose=0).reshape(-1)
    prediction = (probability >= 0.5).astype(np.int8)
    return (binary_y(evaluated), prediction, probability, evaluated)


In [11]:
def build_wang_lstm(n_features):
    model_input = tf.keras.layers.Input(shape=(1, n_features))
    x = model_input
    for _ in range(4):
        x = tf.keras.layers.LSTM(128, return_sequences=True)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.LSTM(256, return_sequences=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(model_input, output)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    return model

def fit_wang_lstm(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_wang_preprocessor(train_frame)
    model = build_wang_lstm(X_train.shape[1])
    model.fit(X_train[:, np.newaxis, :], binary_y(train_frame), validation_split=0.1, batch_size=BATCH_SIZES['wang_lstm'], epochs=EPOCHS['wang_lstm'], shuffle=True, callbacks=standard_callbacks(patience=5), verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_wang_lstm(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_wang_preprocessor(fitted['preprocessor'], evaluated)
    probability = fitted['model'].predict(X_test[:, np.newaxis, :], batch_size=BATCH_SIZES['wang_lstm'], verbose=0).reshape(-1)
    prediction = (probability >= 0.5).astype(np.int8)
    return (binary_y(evaluated), prediction, probability, evaluated)


In [12]:
def fit_anwar_preprocessor(train_frame):
    matrix = numeric_matrix(train_frame, CSE2018_FEATURE_COLUMNS)
    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)
    retained_features = [column for column in CSE2018_FEATURE_COLUMNS if matrix[column].nunique(dropna=False) > 1]
    correlations = matrix[retained_features].corr(method='pearson').abs()
    upper_triangle = correlations.where(np.triu(np.ones(correlations.shape), k=1).astype(bool))
    correlated_to_remove = [column for column in upper_triangle.columns if (upper_triangle[column] > 0.95).any()]
    retained_features = [column for column in retained_features if column not in correlated_to_remove]
    scaler = MinMaxScaler(feature_range=(0, 1))
    transformed = scaler.fit_transform(matrix[retained_features]).astype(np.float32)
    transformed = np.clip(transformed, 0.0, 1.0)
    return ({'feature_columns': retained_features, 'medians': medians, 'scaler': scaler, 'correlated_removed': correlated_to_remove}, transformed)

def transform_anwar_preprocessor(fitted, frame):
    matrix = numeric_matrix(frame, fitted['feature_columns'])
    matrix = matrix.fillna(fitted['medians'].reindex(fitted['feature_columns']).fillna(0.0))
    transformed = fitted['scaler'].transform(matrix).astype(np.float32)
    return np.clip(transformed, 0.0, 1.0)

def build_anwar_lightweight_1dcnn(n_features):
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(n_features, 1)), tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.Conv1D(filters=16, kernel_size=3, padding='same', activation='relu'), tf.keras.layers.BatchNormalization(), tf.keras.layers.GlobalAveragePooling1D(), tf.keras.layers.Dropout(0.3), tf.keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=tf.keras.losses.BinaryCrossentropy(), metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'), tf.keras.metrics.AUC(name='auc')])
    return model

def select_balanced_accuracy_threshold(y_true, probabilities):
    candidate_thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)
    scores = []
    for threshold in candidate_thresholds:
        prediction = (probabilities >= threshold).astype(np.int8)
        scores.append(balanced_accuracy_score(y_true, prediction))
    return float(candidate_thresholds[int(np.argmax(scores))])

def fit_anwar_lightweight_1dcnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    train_part, validation_part = train_test_split(train_frame, test_size=0.1, random_state=seed, shuffle=True, stratify=train_frame[BINARY_COLUMN])
    preprocessor, X_train = fit_anwar_preprocessor(train_part)
    X_validation = transform_anwar_preprocessor(preprocessor, validation_part)
    y_train = binary_y(train_part)
    y_validation = binary_y(validation_part)
    class_weight_values = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train)
    class_weight = {0: float(class_weight_values[0]), 1: float(class_weight_values[1])}
    training_counts = np.bincount(y_train, minlength=2)
    minority_ratio = training_counts.min() / training_counts.max()
    if minority_ratio < 0.25 and len(X_train) < 1000000 and (training_counts.min() >= 6):
        X_train, y_train = SMOTE(random_state=seed).fit_resample(X_train, y_train)
    model = build_anwar_lightweight_1dcnn(X_train.shape[1])
    model.fit(X_train[..., np.newaxis], y_train.astype(np.float32), validation_data=(X_validation[..., np.newaxis], y_validation.astype(np.float32)), batch_size=BATCH_SIZES['anwar_lightweight_1dcnn'], epochs=EPOCHS['anwar_lightweight_1dcnn'], shuffle=True, class_weight=class_weight, callbacks=standard_callbacks(monitor='val_auc', mode='max', patience=4), verbose=2)
    validation_probability = model.predict(X_validation[..., np.newaxis], batch_size=BATCH_SIZES['anwar_lightweight_1dcnn'], verbose=0).reshape(-1)
    threshold = select_balanced_accuracy_threshold(y_validation, validation_probability)
    return {'model': model, 'preprocessor': preprocessor, 'threshold': threshold, 'seed': seed}

def predict_anwar_lightweight_1dcnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_anwar_preprocessor(fitted['preprocessor'], evaluated)
    probability = fitted['model'].predict(X_test[..., np.newaxis], batch_size=BATCH_SIZES['anwar_lightweight_1dcnn'], verbose=0).reshape(-1)
    prediction = (probability >= fitted['threshold']).astype(np.int8)
    return (binary_y(evaluated), prediction, probability, evaluated)


In [13]:
def fit_ensemble_feature_rf(train_frame, seed):
    set_all_seeds(seed)
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_numeric_preprocessor(train_frame, CSE2018_FEATURE_COLUMNS, scaler_kind='standard', drop_constant=True)
    y_train = binary_y(train_frame)
    feature_names = np.array(preprocessor['feature_columns'])
    training_frame = pd.DataFrame(X_train, columns=feature_names)
    training_frame[BINARY_COLUMN] = y_train
    target_correlations = training_frame.corr(numeric_only=True)[BINARY_COLUMN].drop(BINARY_COLUMN).abs()
    correlation_features = set(target_correlations[target_correlations > 0.5].index)
    mutual_information_k = min(20, X_train.shape[1])
    mi_selector = SelectKBest(score_func=mutual_info_classif, k=mutual_information_k)
    mi_selector.fit(X_train, y_train)
    mi_features = set(feature_names[mi_selector.get_support()])
    pca_model = PCA(n_components=0.95, svd_solver='full')
    pca_model.fit(X_train)
    dominant_indices = np.unique(np.argmax(np.abs(pca_model.components_), axis=1))
    pca_features = set(feature_names[dominant_indices])
    selected_features = sorted(correlation_features | mi_features | pca_features)
    if not selected_features:
        raise ValueError('Feature-selection union is empty.')
    selected_indices = np.array([int(np.where(feature_names == feature)[0][0]) for feature in selected_features])
    X_selected = X_train[:, selected_indices]
    model = RandomForestClassifier(n_estimators=10, max_depth=None, criterion='gini', min_samples_split=2, min_samples_leaf=1, max_features='sqrt', bootstrap=True, random_state=seed, n_jobs=-1)
    model.fit(X_selected, y_train)
    return {'model': model, 'preprocessor': preprocessor, 'selected_features': selected_features, 'selected_indices': selected_indices, 'seed': seed}

def predict_ensemble_feature_rf(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_numeric_preprocessor(fitted['preprocessor'], evaluated)
    X_selected = X_test[:, fitted['selected_indices']]
    prediction = fitted['model'].predict(X_selected).astype(np.int8)
    probability = fitted['model'].predict_proba(X_selected)[:, 1]
    return (binary_y(evaluated), prediction, probability, evaluated)


In [14]:
MODEL_REGISTRY = {'alzughabi_pso_dnn': {'fit': fit_alzughabi_pso_dnn, 'predict': predict_alzughabi_pso_dnn}, 'wang_cnn': {'fit': fit_wang_cnn, 'predict': predict_wang_cnn}, 'wang_rnn': {'fit': fit_wang_rnn, 'predict': predict_wang_rnn}, 'wang_lstm': {'fit': fit_wang_lstm, 'predict': predict_wang_lstm}, 'anwar_lightweight_1dcnn': {'fit': fit_anwar_lightweight_1dcnn, 'predict': predict_anwar_lightweight_1dcnn}, 'ensemble_feature_rf': {'fit': fit_ensemble_feature_rf, 'predict': predict_ensemble_feature_rf}}
missing_models = [model for model in MODEL_KEYS if model not in MODEL_REGISTRY]
if missing_models:
    raise KeyError(f'Unregistered models: {missing_models}')
print('Registered models:', list(MODEL_REGISTRY))


Registered models: ['alzughabi_pso_dnn', 'wang_cnn', 'wang_rnn', 'wang_lstm', 'anwar_lightweight_1dcnn', 'ensemble_feature_rf']


In [15]:
MODEL_TEST_RATIO_COUNTS = {'alzughabi_pso_dnn': {'normal': 4005868, 'attack': 706382}, 'wang_cnn': {'normal': 1898660, 'attack': 265173}, 'wang_rnn': {'normal': 1898660, 'attack': 265173}, 'wang_lstm': {'normal': 1898660, 'attack': 265173}, 'anwar_lightweight_1dcnn': {'normal': 2070696, 'attack': 264868}, 'ensemble_feature_rf': {'normal': 185932, 'attack': 23783}}
MODEL_TEST_RATIOS = {}
for model_key, counts in MODEL_TEST_RATIO_COUNTS.items():
    total = counts['normal'] + counts['attack']
    MODEL_TEST_RATIOS[model_key] = {'normal_ratio': counts['normal'] / total, 'attack_ratio': counts['attack'] / total, 'original_test_normal': counts['normal'], 'original_test_attack': counts['attack'], 'original_test_total': total}
model_ratio_table = pd.DataFrame([{'model': model_key, 'original_test_normal': values['original_test_normal'], 'original_test_attack': values['original_test_attack'], 'target_normal_pct': 100 * values['normal_ratio'], 'target_attack_pct': 100 * values['attack_ratio']} for model_key, values in MODEL_TEST_RATIOS.items()])

In [ ]:
PARTITION_DIR = Path('/content/drive/MyDrive/Array_submission/Prepared_partitions/splits') / 'cicflow_cic18'
mixed_train_reference = pd.read_csv(PARTITION_DIR / 'balanced_train.csv', low_memory=False)
eligible_test_reference = pd.read_csv(PARTITION_DIR / 'test_all_sources.csv', low_memory=False)
required_columns = set(CSE2018_FEATURE_COLUMNS + [SOURCE_COLUMN, LABEL_COLUMN, BINARY_COLUMN, ORIGINAL_LABEL_COLUMN])
for frame_name, frame in [('balanced_train', mixed_train_reference), ('test_all_sources', eligible_test_reference)]:
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise KeyError(f'{frame_name} is missing columns: {missing}')
source_class_counts = mixed_train_reference.groupby([SOURCE_COLUMN, BINARY_COLUMN]).size()
BALANCE_TARGET_PER_CLASS_SOURCE = int(source_class_counts.min())
if not source_class_counts.eq(BALANCE_TARGET_PER_CLASS_SOURCE).all():
    raise AssertionError(f'Loaded training partition is not source/class balanced: {source_class_counts.to_dict()}')
mixed_test_sets = {source: part.reset_index(drop=True) for source, part in eligible_test_reference.groupby(SOURCE_COLUMN, sort=True)}
partition_audit = pd.read_csv(PARTITION_DIR / 'partition_manifest.csv')
training_distribution_export = pd.read_csv(PARTITION_DIR / 'balanced_training_distribution.csv')
split_manifest = pd.DataFrame([{'dataset': 'balanced_training_partition', 'rows': len(mixed_train_reference)}, {'dataset': 'test_partition_before_ratio_matching', 'rows': len(eligible_test_reference)}])
split_manifest.to_csv(RESULTS_DIR / 'mixed_source_split_manifest.csv', index=False)
partition_audit.to_csv(RESULTS_DIR / 'partition_audit.csv', index=False)
training_distribution_export.to_csv(RESULTS_DIR / 'balanced_training_distribution.csv', index=False)
eligible_test_reference.to_csv(RESULTS_DIR / 'test_all_sources_before_ratio_matching.csv', index=False)
print('Loaded balanced training set:', mixed_train_reference.shape)
print('Loaded test partition:', eligible_test_reference.shape)


In [ ]:
def choose_binary_sample_counts(normal_available, attack_available, target_normal_ratio):
    normal_available = int(normal_available)
    attack_available = int(attack_available)
    target_normal_ratio = float(target_normal_ratio)
    target_attack_ratio = 1.0 - target_normal_ratio
    if normal_available < 1 or attack_available < 1:
        raise ValueError('Ratio matching requires both normal and attack rows.')
    if not 0.0 < target_normal_ratio < 1.0:
        raise ValueError('Target normal ratio must be strictly between 0 and 1.')
    candidates = set()

    def add_candidate(normal_count, attack_count):
        normal_count = int(normal_count)
        attack_count = int(attack_count)
        if normal_count >= 1 and attack_count >= 1 and (normal_count <= normal_available) and (attack_count <= attack_available):
            candidates.add((normal_count, attack_count))
    ideal_attack_from_all_normal = normal_available * target_attack_ratio / target_normal_ratio
    for attack_count in {math.floor(ideal_attack_from_all_normal), round(ideal_attack_from_all_normal), math.ceil(ideal_attack_from_all_normal)}:
        add_candidate(normal_available, attack_count)
    ideal_normal_from_all_attack = attack_available * target_normal_ratio / target_attack_ratio
    for normal_count in {math.floor(ideal_normal_from_all_attack), round(ideal_normal_from_all_attack), math.ceil(ideal_normal_from_all_attack)}:
        add_candidate(normal_count, attack_available)
    if not candidates:
        raise ValueError('No feasible two-class sample could be allocated.')

    def candidate_key(pair):
        normal_count, attack_count = pair
        total = normal_count + attack_count
        achieved = normal_count / total
        error = abs(achieved - target_normal_ratio)
        return (-total, error)
    return min(candidates, key=candidate_key)

def proportional_group_sample(frame, sample_size, group_column, seed):
    """Sample a requested total while approximately preserving group shares."""
    sample_size = int(sample_size)
    if sample_size < 0 or sample_size > len(frame):
        raise ValueError('Requested grouped sample size is infeasible.')
    if sample_size == len(frame):
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)
    if sample_size == 0:
        return frame.iloc[0:0].copy()
    working = frame.copy()
    working['_ratio_group'] = working[group_column].astype('string').fillna('missing')
    counts = working['_ratio_group'].value_counts(sort=False).sort_index()
    ideal = counts.astype(float) / counts.sum() * sample_size
    allocation = np.floor(ideal).astype(int)
    allocation = pd.Series(np.minimum(allocation.to_numpy(), counts.to_numpy()), index=counts.index, dtype=int)
    remaining = sample_size - int(allocation.sum())
    fractional = ideal - np.floor(ideal)
    priority = sorted(counts.index, key=lambda group: (-float(fractional.loc[group]), str(group)))
    while remaining > 0:
        made_progress = False
        for group in priority:
            if allocation.loc[group] < counts.loc[group]:
                allocation.loc[group] += 1
                remaining -= 1
                made_progress = True
                if remaining == 0:
                    break
        if not made_progress:
            raise RuntimeError('Could not distribute grouped sample remainder.')
    parts = []
    for offset, group in enumerate(allocation.index):
        take = int(allocation.loc[group])
        if take == 0:
            continue
        group_frame = working[working['_ratio_group'].eq(group)].drop(columns=['_ratio_group'])
        parts.append(group_frame.sample(n=take, replace=False, random_state=seed + offset + 1))
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

def match_model_test_ratio(frame, model_key, seed):
    ratio = MODEL_TEST_RATIOS[model_key]
    target_normal_ratio = ratio['normal_ratio']
    normal_frame = frame[frame[BINARY_COLUMN].eq(0)]
    attack_frame = frame[frame[BINARY_COLUMN].eq(1)]
    normal_selected, attack_selected = choose_binary_sample_counts(len(normal_frame), len(attack_frame), target_normal_ratio)
    sampled_normal = normal_frame.sample(n=normal_selected, replace=False, random_state=seed)
    sampled_attack = proportional_group_sample(attack_frame, attack_selected, LABEL_COLUMN, seed + 10000)
    matched = pd.concat([sampled_normal, sampled_attack], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    actual_normal = int((matched[BINARY_COLUMN] == 0).sum())
    actual_attack = int((matched[BINARY_COLUMN] == 1).sum())
    actual_total = len(matched)
    audit = {'model': model_key, 'available_rows': len(frame), 'available_normal': len(normal_frame), 'available_attack': len(attack_frame), 'selected_rows': actual_total, 'selected_normal': actual_normal, 'selected_attack': actual_attack, 'target_normal_pct': 100 * target_normal_ratio, 'target_attack_pct': 100 * (1.0 - target_normal_ratio), 'actual_normal_pct': 100 * actual_normal / actual_total, 'actual_attack_pct': 100 * actual_attack / actual_total}
    return (matched, audit)
model_matched_test_sets = {}
model_test_ratio_audits = {}
model_test_ratio_manifest_rows = []
for model_offset, model_key in enumerate(MODEL_KEYS):
    model_matched_test_sets[model_key] = {}
    for source_offset, (test_source, raw_test_frame) in enumerate(mixed_test_sets.items()):
        matched, audit = match_model_test_ratio(raw_test_frame, model_key=model_key, seed=PARTITION_SEED + 1000 * model_offset + source_offset)
        model_matched_test_sets[model_key][test_source] = matched
        model_test_ratio_audits[model_key, test_source] = audit
        model_test_ratio_manifest_rows.append({'model': model_key, 'test_source': test_source, **audit})
        if SAVE_TEST_SETS:
            matched.to_csv(RESULTS_DIR / f'test_{model_key}_{test_source}.csv', index=False)
    if SAVE_TEST_SETS:
        pd.concat(model_matched_test_sets[model_key].values(), ignore_index=True).to_csv(RESULTS_DIR / f'test_{model_key}_all_sources.csv', index=False)
model_test_ratio_manifest = pd.DataFrame(model_test_ratio_manifest_rows)
model_test_ratio_manifest.to_csv(RESULTS_DIR / 'model_test_ratio_manifest.csv', index=False)


In [18]:
def merge_and_save(path, new_rows, key_columns):
    path = Path(path)
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_rows], ignore_index=True)
    else:
        combined = new_rows.copy()
    combined = combined.drop_duplicates(subset=key_columns, keep='last').reset_index(drop=True)
    combined.to_csv(path, index=False)
    return combined


In [19]:
MIXED_EXPERIMENT = 'mixed_source_cic18'
TRAINING_VARIANT = 'mixed_source_balanced_65120'

def evaluate_model_specific_ratio_experiment(model_key, seed):
    adapter = MODEL_REGISTRY[model_key]
    train_frame = mixed_train_reference
    source_class_quota = BALANCE_TARGET_PER_CLASS_SOURCE
    target = MODEL_TEST_RATIOS[model_key]
    print(f'\n{MIXED_EXPERIMENT} | {model_key} | seed={seed}', flush=True)
    print(f'  Target test prevalence: {100 * target['normal_ratio']:.6f}% normal, {100 * target['attack_ratio']:.6f}% attack', flush=True)
    print(f'  Balanced training rows: {len(train_frame):,}; quota={source_class_quota:,}', flush=True)
    fit_started = time.perf_counter()
    fitted = adapter['fit'](train_frame, seed)
    fit_seconds = time.perf_counter() - fit_started
    overall_rows = []
    category_frames = []
    subtype_frames = []
    ratio_rows = []
    pooled_true = []
    pooled_prediction = []
    pooled_score = []
    pooled_evaluated = []
    for source_offset, (test_source, matched_test_frame) in enumerate(model_matched_test_sets[model_key].items()):
        raw_test_frame = mixed_test_sets[test_source]
        ratio_audit = model_test_ratio_audits[model_key, test_source].copy()
        ratio_audit.update({'experiment': MIXED_EXPERIMENT, 'seed': seed, 'test_source': test_source})
        ratio_rows.append(ratio_audit)
        print(f'  Testing {test_source}: {len(raw_test_frame):,} available -> {len(matched_test_frame):,} matched; normal={ratio_audit['actual_normal_pct']:.6f}%', flush=True)
        prediction_started = time.perf_counter()
        y_true, y_pred, y_score, evaluated = adapter['predict'](fitted, matched_test_frame)
        prediction_seconds = time.perf_counter() - prediction_started
        metrics = calculate_binary_metrics(y_true, y_pred, y_score)
        evaluated_normal = int(np.sum(y_true == 0))
        evaluated_attack = int(np.sum(y_true == 1))
        evaluated_total = len(y_true)
        overall_rows.append({'experiment': MIXED_EXPERIMENT, 'training_variant': TRAINING_VARIANT, 'model': model_key, 'seed': seed, 'test_source': test_source, 'source_class_quota': source_class_quota, 'n_train_used': len(train_frame), 'n_test_raw': len(raw_test_frame), 'n_test_ratio_matched': len(matched_test_frame), 'n_test_evaluated': evaluated_total, 'target_normal_pct': 100 * target['normal_ratio'], 'target_attack_pct': 100 * target['attack_ratio'], 'actual_normal_pct': 100 * evaluated_normal / evaluated_total, 'actual_attack_pct': 100 * evaluated_attack / evaluated_total, 'fit_seconds': fit_seconds, 'prediction_seconds': prediction_seconds, **metrics})
        category_table = category_recall_table(evaluated, y_true, y_pred)
        category_table['experiment'] = MIXED_EXPERIMENT
        category_table['training_variant'] = TRAINING_VARIANT
        category_table['model'] = model_key
        category_table['seed'] = seed
        category_table['test_source'] = test_source
        category_frames.append(category_table)
        subtype_table = subtype_recall_table(evaluated, y_true, y_pred)
        subtype_table['experiment'] = MIXED_EXPERIMENT
        subtype_table['training_variant'] = TRAINING_VARIANT
        subtype_table['model'] = model_key
        subtype_table['seed'] = seed
        subtype_table['test_source'] = test_source
        subtype_frames.append(subtype_table)
        pooled_true.append(y_true)
        pooled_prediction.append(y_pred)
        pooled_score.append(y_score)
        pooled_evaluated.append(evaluated)
    pooled_true_array = np.concatenate(pooled_true)
    pooled_prediction_array = np.concatenate(pooled_prediction)
    pooled_score_array = np.concatenate(pooled_score)
    pooled_metrics = calculate_binary_metrics(pooled_true_array, pooled_prediction_array, pooled_score_array)
    pooled_normal = int(np.sum(pooled_true_array == 0))
    pooled_attack = int(np.sum(pooled_true_array == 1))
    pooled_total = len(pooled_true_array)
    overall_rows.append({'experiment': MIXED_EXPERIMENT, 'training_variant': TRAINING_VARIANT, 'model': model_key, 'seed': seed, 'test_source': 'POOLED_ALL_ROWS', 'source_class_quota': source_class_quota, 'n_train_used': len(train_frame), 'n_test_raw': int(sum((len(frame) for frame in mixed_test_sets.values()))), 'n_test_ratio_matched': pooled_total, 'n_test_evaluated': pooled_total, 'target_normal_pct': 100 * target['normal_ratio'], 'target_attack_pct': 100 * target['attack_ratio'], 'actual_normal_pct': 100 * pooled_normal / pooled_total, 'actual_attack_pct': 100 * pooled_attack / pooled_total, 'fit_seconds': fit_seconds, 'prediction_seconds': np.nan, **pooled_metrics})
    ratio_rows.append({'experiment': MIXED_EXPERIMENT, 'model': model_key, 'seed': seed, 'test_source': 'POOLED_ALL_ROWS', 'available_rows': int(sum((len(frame) for frame in mixed_test_sets.values()))), 'available_normal': int(sum(((frame[BINARY_COLUMN] == 0).sum() for frame in mixed_test_sets.values()))), 'available_attack': int(sum(((frame[BINARY_COLUMN] == 1).sum() for frame in mixed_test_sets.values()))), 'selected_rows': pooled_total, 'selected_normal': pooled_normal, 'selected_attack': pooled_attack, 'target_normal_pct': 100 * target['normal_ratio'], 'target_attack_pct': 100 * target['attack_ratio'], 'actual_normal_pct': 100 * pooled_normal / pooled_total, 'actual_attack_pct': 100 * pooled_attack / pooled_total})
    del fitted
    gc.collect()
    tf.keras.backend.clear_session()
    return (pd.DataFrame(overall_rows), pd.concat(category_frames, ignore_index=True), pd.concat(subtype_frames, ignore_index=True), pd.DataFrame(ratio_rows))


In [20]:
def run_model_specific_ratio_experiment():
    overall_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_all_runs_partial.csv'
    category_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_category_runs_partial.csv'
    subtype_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_subtype_runs_partial.csv'
    ratio_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_ratio_manifest_partial.csv'
    completed = pd.read_csv(overall_path) if overall_path.exists() else pd.DataFrame()
    overall_accumulated = completed.copy()
    category_accumulated = pd.read_csv(category_path) if category_path.exists() else pd.DataFrame()
    subtype_accumulated = pd.read_csv(subtype_path) if subtype_path.exists() else pd.DataFrame()
    ratio_accumulated = pd.read_csv(ratio_path) if ratio_path.exists() else pd.DataFrame()
    for model_key in MODEL_KEYS:
        for seed in SEEDS:
            if not completed.empty:
                complete_mask = completed['experiment'].eq(MIXED_EXPERIMENT) & completed['model'].eq(model_key) & completed['seed'].eq(seed) & completed['test_source'].eq('POOLED_ALL_ROWS')
                is_complete = bool(complete_mask.any())
            else:
                is_complete = False
            if is_complete:
                print(f'SKIP completed: {model_key}, seed={seed}', flush=True)
                continue
            overall, categories, subtypes, ratios = evaluate_model_specific_ratio_experiment(model_key, seed)
            category_accumulated = merge_and_save(category_path, categories, ['experiment', 'training_variant', 'model', 'seed', 'test_source', 'test_category'])
            subtype_accumulated = merge_and_save(subtype_path, subtypes, ['experiment', 'training_variant', 'model', 'seed', 'test_source', 'test_subtype'])
            ratio_accumulated = merge_and_save(ratio_path, ratios, ['experiment', 'model', 'seed', 'test_source'])
            overall_accumulated = merge_and_save(overall_path, overall, ['experiment', 'training_variant', 'model', 'seed', 'test_source'])
            completed = overall_accumulated
    return (overall_accumulated, category_accumulated, subtype_accumulated, ratio_accumulated)


In [21]:
exp4_overall, exp4_categories, exp4_subtypes, exp4_ratio_manifest = run_model_specific_ratio_experiment()



mixed_source_cic18 | alzughabi_pso_dnn | seed=11
  Target test prevalence: 85.009666% normal, 14.990334% attack
  Balanced training rows: 65,120; quota=8,140
Epoch 1/100
29/29 - 11s - 373ms/step - accuracy: 0.7015 - loss: 0.6370 - val_accuracy: 0.7004 - val_loss: 0.6041
Epoch 2/100
29/29 - 0s - 6ms/step - accuracy: 0.7793 - loss: 0.5813 - val_accuracy: 0.8061 - val_loss: 0.5561
Epoch 3/100
29/29 - 0s - 6ms/step - accuracy: 0.8073 - loss: 0.5396 - val_accuracy: 0.8100 - val_loss: 0.5185
Epoch 4/100
29/29 - 0s - 6ms/step - accuracy: 0.8115 - loss: 0.5068 - val_accuracy: 0.8159 - val_loss: 0.4878
Epoch 5/100
29/29 - 0s - 6ms/step - accuracy: 0.8186 - loss: 0.4821 - val_accuracy: 0.8348 - val_loss: 0.4657
Epoch 6/100
29/29 - 0s - 6ms/step - accuracy: 0.8362 - loss: 0.4641 - val_accuracy: 0.8431 - val_loss: 0.4488
Epoch 7/100
29/29 - 0s - 6ms/step - accuracy: 0.8390 - loss: 0.4498 - val_accuracy: 0.8455 - val_loss: 0.4351
Epoch 8/100
29/29 - 0s - 6ms/step - accuracy: 0.8406 - loss: 0.4370 

  Testing CSE-CIC-IDS2018: 6,978 available -> 3,977 matched; normal=87.729444%
  Testing DARPA: 6,978 available -> 3,977 matched; normal=87.729444%
  Testing UNSW-NB15: 6,978 available -> 3,977 matched; normal=87.729444%

mixed_source_cic18 | wang_cnn | seed=33
  Target test prevalence: 87.745219% normal, 12.254781% attack
  Balanced training rows: 65,120; quota=8,140
Epoch 1/50
29/29 - 10s - 339ms/step - accuracy: 0.8037 - loss: 0.4851 - val_accuracy: 0.8365 - val_loss: 0.4127 - learning_rate: 0.0100
Epoch 2/50
29/29 - 0s - 11ms/step - accuracy: 0.8897 - loss: 0.2768 - val_accuracy: 0.7502 - val_loss: 0.4783 - learning_rate: 0.0100
Epoch 3/50
29/29 - 0s - 11ms/step - accuracy: 0.9020 - loss: 0.2358 - val_accuracy: 0.9100 - val_loss: 0.3093 - learning_rate: 0.0100
Epoch 4/50
29/29 - 0s - 10ms/step - accuracy: 0.9072 - loss: 0.2179 - val_accuracy: 0.8990 - val_loss: 0.3229 - learning_rate: 0.0100
Epoch 5/50
29/29 - 0s - 11ms/step - accuracy: 0.9083 - loss: 0.2116 - val_accuracy: 0.8833 

  Testing CSE-CIC-IDS2018: 6,978 available -> 3,977 matched; normal=87.729444%
  Testing DARPA: 6,978 available -> 3,977 matched; normal=87.729444%
  Testing UNSW-NB15: 6,978 available -> 3,977 matched; normal=87.729444%

mixed_source_cic18 | wang_cnn | seed=44
  Target test prevalence: 87.745219% normal, 12.254781% attack
  Balanced training rows: 65,120; quota=8,140
Epoch 1/50
29/29 - 10s - 351ms/step - accuracy: 0.7633 - loss: 0.5787 - val_accuracy: 0.8452 - val_loss: 0.4134 - learning_rate: 0.0100
Epoch 2/50
29/29 - 0s - 12ms/step - accuracy: 0.8753 - loss: 0.3053 - val_accuracy: 0.7898 - val_loss: 0.4149 - learning_rate: 0.0100
Epoch 3/50

Epoch 3: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
29/29 - 0s - 11ms/step - accuracy: 0.8948 - loss: 0.2620 - val_accuracy: 0.7918 - val_loss: 0.4214 - learning_rate: 0.0100
Epoch 4/50
29/29 - 0s - 11ms/step - accuracy: 0.9057 - loss: 0.2287 - val_accuracy: 0.9017 - val_loss: 0.2588 - learning_rate: 0.0050
Epoch 5/50
29/2

In [22]:
REPORT_METRICS = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'roc_auc']
exp4_per_source_summary = exp4_overall.groupby(['experiment', 'training_variant', 'model', 'test_source'], dropna=False)[REPORT_METRICS].agg(['mean', 'std', 'min', 'max']).reset_index()
exp4_source_macro_by_seed = exp4_overall[exp4_overall['test_source'].ne('POOLED_ALL_ROWS')].groupby(['experiment', 'training_variant', 'model', 'seed'], dropna=False).agg(source_macro_f1=('f1', 'mean'), source_macro_balanced_accuracy=('balanced_accuracy', 'mean'), source_macro_recall=('recall', 'mean'), source_macro_specificity=('specificity', 'mean')).reset_index()
exp4_source_macro_summary = exp4_source_macro_by_seed.groupby(['experiment', 'training_variant', 'model'], dropna=False)[['source_macro_f1', 'source_macro_balanced_accuracy', 'source_macro_recall', 'source_macro_specificity']].agg(['mean', 'std', 'min', 'max']).reset_index()
exp4_category_summary = exp4_categories.groupby(['experiment', 'training_variant', 'model', 'test_source', 'test_category'], dropna=False).agg(support=('support', 'mean'), attack_recall_mean=('attack_recall', 'mean'), attack_recall_std=('attack_recall', 'std'), benign_specificity_mean=('benign_specificity', 'mean'), benign_specificity_std=('benign_specificity', 'std')).reset_index()
exp4_ratio_summary = exp4_ratio_manifest.groupby(['experiment', 'model', 'test_source'], dropna=False).agg(target_normal_pct=('target_normal_pct', 'first'), target_attack_pct=('target_attack_pct', 'first'), selected_rows_mean=('selected_rows', 'mean'), actual_normal_pct_mean=('actual_normal_pct', 'mean'), actual_normal_pct_min=('actual_normal_pct', 'min'), actual_normal_pct_max=('actual_normal_pct', 'max'), actual_attack_pct_mean=('actual_attack_pct', 'mean')).reset_index()
exports = {f'{MIXED_EXPERIMENT}_all_runs.csv': exp4_overall, f'{MIXED_EXPERIMENT}_category_runs.csv': exp4_categories, f'{MIXED_EXPERIMENT}_subtype_runs.csv': exp4_subtypes, f'{MIXED_EXPERIMENT}_ratio_manifest.csv': exp4_ratio_manifest, f'{MIXED_EXPERIMENT}_per_source_summary.csv': exp4_per_source_summary, f'{MIXED_EXPERIMENT}_source_macro_by_seed.csv': exp4_source_macro_by_seed, f'{MIXED_EXPERIMENT}_source_macro_summary.csv': exp4_source_macro_summary, f'{MIXED_EXPERIMENT}_category_summary.csv': exp4_category_summary, f'{MIXED_EXPERIMENT}_ratio_summary.csv': exp4_ratio_summary}
for filename, frame in exports.items():
    frame.to_csv(RESULTS_DIR / filename, index=False)
result_columns = ['model', 'seed', 'test_source', 'source_class_quota', 'n_train_used', 'n_test_raw', 'n_test_ratio_matched', 'n_test_evaluated', 'target_normal_pct', 'target_attack_pct', 'actual_normal_pct', 'actual_attack_pct', 'fit_seconds', 'prediction_seconds', 'n_test', 'tn', 'fp', 'fn', 'tp', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'roc_auc']
source_order = ['CIC-IDS2017', 'CSE-CIC-IDS2018', 'DARPA', 'UNSW-NB15', 'POOLED_ALL_ROWS']
table_results = exp4_overall.copy()
table_results['__model'] = pd.Categorical(table_results['model'], categories=MODEL_KEYS, ordered=True)
table_results['__seed'] = pd.Categorical(table_results['seed'], categories=SEEDS, ordered=True)
table_results['__source'] = pd.Categorical(table_results['test_source'], categories=source_order, ordered=True)
table_results = table_results.sort_values(['__model', '__seed', '__source']).loc[:, result_columns]
expected_result_rows = len(MODEL_KEYS) * len(SEEDS) * len(source_order)
if len(table_results) != expected_result_rows:
    raise AssertionError(f'Expected {expected_result_rows} CIC18 result rows, got {len(table_results)}.')
table_results.to_csv(RESULTS_DIR / 'all_runs_CIC18.csv', index=False)